# 03 — Train action-aware multimodal two-tower

Pipeline Coveo v1 — mọi output được version hóa và không ghi đè.

In [ ]:
%pip install -q -e ".[coveo]"

In [ ]:
from pathlib import Path
import os, json, yaml

def find_repo():
    here = Path.cwd().resolve()
    for root in (here, *here.parents):
        if (root / 'pyproject.toml').exists(): return root
    raise FileNotFoundError('Không tìm thấy pyproject.toml')

REPO = find_repo()
os.chdir(REPO)
cfg = yaml.safe_load((REPO / 'configs/coveo.yaml').read_text(encoding='utf-8'))
PROFILE = os.getenv('COVEO_PROFILE', cfg['project']['profile'])
print('repo=', REPO, 'profile=', PROFILE)

In [ ]:
from datn.recommenders.coveo.pipeline import RetrievalTrainConfig, train_retrieval
r = cfg['retrieval']
params = RetrievalTrainConfig(max_seq_len=r['max_seq_len'], d_model=r['d_model'], n_heads=r['n_heads'],
    n_layers=r['n_layers'], dropout=r['dropout'], batch_size=r['batch_size'], epochs=r[f'epochs_{PROFILE}'],
    lr=r['lr'], weight_decay=r['weight_decay'], purchase_alpha=r['purchase_alpha'], temperature=r['temperature'],
    patience=r['patience'], seed=cfg['project']['seed'], device=r['device'])
metrics = train_retrieval(REPO / cfg['paths']['processed_dir'], REPO / cfg['paths']['embeddings_dir'],
    REPO / cfg['paths']['retrieval_dir'], params)
display(metrics)

Checkpoint tốt nhất được chọn duy nhất bằng `validation HitRate@1000`; test chỉ chạy sau khi khóa checkpoint.